# Embed & cluster the admission-medication column

This notebook embeds the admission-medication text locally and clusters it with
KMeans, so you can draw a stratified sample for annotation.

**Pipeline:** `encode` (dense embeddings) -> L2 normalize -> PCA -> L2 normalize -> KMeans

Everything runs on your machine. Your clinical text never leaves the computer.

---

### Before you start: download the model once (setup step, *outside* this notebook)

The only network access in this whole workflow is the one-time model download.
Do it once on a machine with internet, then this notebook runs fully offline:

```bash
uvx hf download sentence-transformers/all-MiniLM-L6-v2

```

This caches the weights (~80 MB) under `~/.cache/huggingface/`. If your analysis
environment is air-gapped, run the command on a connected machine and copy the
cache over. After that, the cells below never touch the network.

## 1. Import & Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

In [ ]:
load_dotenv()

### 1.1. Force Offline Mode

These env vars tell the Hugging Face libraries: *use only the local cache, never
the network.* If the model is cached, it loads normally; if it isn't, you get an
error instead of a silent download. That turns "nothing hits the network" into a
guarantee you can verify.

**This must be the first cell** — the libraries read these vars at import time, so
they have to be set before any huggingface import.

In [ ]:
# Block both HF layers: the hub client and the transformers library.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("Offline mode set. The model must already be cached (see setup step above).")



## 2. Local Config

One place for the knobs you'll actually touch.
Fix the seeds: 
- PCA and KMeans are
otherwise non-deterministic.

### 2.1. Local Variables

In [ ]:
DATASET_PATH = f'{os.environ["REPO_PATH"]}/data/datasets/structured_extraction/medication_on_admission/medication_on_admission_final_dataset.parquet'
TEXT_COLUMN  = "meds_on_admission_cleaned"   # the medication column to embed


In [ ]:
output_data_path = f'{os.environ["REPO_PATH"]}/scripts/3_information_extraction/medications_on_admission/data'

# Creates the entire folder structure; does nothing if they already exist
os.makedirs(output_data_path, exist_ok=True)


### 2.2. Embed & Cluster Config

In [ ]:
MODEL_NAME_1   = "all-MiniLM-L6-v2"      # a BERT model fine-tuned for sentence embeddings
EMB_CACHE_1    = f'{output_data_path}/{MODEL_NAME_1}_embeddings.npy'        # embeddings are expensive on 312K notes -> cache them

In [ ]:
MODEL_NAME_2   = "all-MiniLM-L6-v2"      # a BERT model fine-tuned for sentence embeddings
EMB_CACHE_2    = f'{output_data_path}/{MODEL_NAME_2}_embeddings.npy'        # embeddings are expensive on 312K notes -> cache them

## 3. Load Dataset

In [ ]:
df = pd.read_parquet(DATASET_PATH)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df1 = df.copy()

df1['meds_on_admission_cleaned_length'] = df1['meds_on_admission_cleaned'].str.len()

df1 = df1.drop(columns = ['meds_on_admission_length'])

df1.info()

In [ ]:
df1['meds_on_admission_cleaned_length'].min()

In [ ]:
texts = df1[TEXT_COLUMN].fillna("").astype(str).tolist()
print(f"Notes to embed: {len(texts)}")

## 4. Load the model (offline)

If this cell errors, the model isn't cached yet — run the `hf download` setup step
first. If it loads cleanly with the offline vars set, that's your proof the
embedding step won't touch the network.

In [ ]:
print("Loading AI model")
print("=" * 50)

# Load model for embeddings
model_1 = SentenceTransformer(MODEL_NAME_1)
print(f"Loaded {MODEL_NAME_1} | embedding dim = {model_1.get_embedding_dimension()}")

In [ ]:
print("Loading AI model")
print("=" * 50)

# Load model for embeddings
model_2 = SentenceTransformer(MODEL_NAME_2)
print(f"Loaded {MODEL_NAME_2} | embedding dim = {model_2.get_embedding_dimension()}")

## 5. Embed

`normalize_embeddings=True` makes each vector unit-length, so KMeans' Euclidean
distance behaves like cosine similarity — the right metric for text embeddings.

We cache to disk so you never re-embed 312K notes between clustering experiments.

In [ ]:
if os.path.exists(EMB_CACHE_1):
    print(f"Loading cached embeddings from {EMB_CACHE_1}")
    emb = np.load(EMB_CACHE_1)
else:
    emb = model_1.encode(
        texts,
        normalize_embeddings=True,   # L2 normalization done here
        batch_size=64,               # raise if you have GPU headroom
        show_progress_bar=True,
    ).astype(np.float32)
    np.save(EMB_CACHE_1, emb)
    print(f"Saved embeddings to {EMB_CACHE_1}")

print(f"Embedding matrix: {emb.shape}")   # (n_notes, 384) for MiniLM

## 6. PCA decision — is it worth reducing dimensions?

**What is PCA?** PCA (Principal Component Analysis) rewrites each embedding
vector using fewer numbers (dimensions) while keeping as much of the variation
as possible. It finds the directions where the data varies most and drops the
near-flat ones, which are usually just noise.

**Why it helps KMeans:** fewer dimensions make KMeans faster, and its distances
stay meaningful — in very high-dimensional space all points drift to roughly
equal distance apart, which blurs the clusters.

**What this cell does:** it does *not* apply PCA — it only diagnoses whether
it's worth it. It checks how many components are needed to keep 95% of the
variance:

- few dimensions hold 95% → embeddings are mostly noise → **apply PCA**
- most dimensions needed → variance is real signal → **skip PCA**

In [ ]:
emb_1 = np.load(f"{output_data_path}/{MODEL_NAME_1}_embeddings.npy")        # cached embeddings (already L2-normalized)
cum = np.cumsum(PCA().fit(emb_1).explained_variance_ratio_)  # cumulative variance per component
n95 = int(np.argmax(cum >= 0.95) + 1)                      # components needed for 95% variance
ratio = n95 / emb_1.shape[1]

print(f"95% variance in {n95}/{emb_1.shape[1]} dims ({ratio:.0%})")
print("APPLY PCA" if ratio < 0.30 else "SKIP PCA" if ratio > 0.65 else "BORDERLINE — test both")

### PCA decision — result: borderline (176/384 dims, 46%)

**176 of 384 (46%) is the ambiguous case** — neither "thin" (where PCA would
strip out a lot of noise) nor "fat" (where PCA would cut nothing useful). Neither
choice is obviously right, so the honest answer is: try both and let the clusters
decide.

**What the number means in practice:** roughly half of the 384 dimensions carry
real signal and the other half is more or less redundant. PCA at 95% would save
~54% of the dimensions (384 → 176) — a decent speed-up, but not a dramatic one.
And since you still need 176 components, you're not clearly in "too many
dimensions" territory, so clustering the full 384 directly is perfectly
defensible too. Hence the tie.

**A caveat that applies especially here:** remember the copy-forward duplicates.
Identical medication lists collapse onto the same point, and large masses of
repeated points can inflate the variance of the first components — which may be
pushing this number down (making it look like fewer dimensions suffice). So the
46% might be slightly optimistic. It doesn't change the "borderline" conclusion,
but it's one more reason not to trust the number alone and to actually look at
the clusters.

### 6.1. Should I use PCA? (empirical check)

The variance diagnostic came back **borderline** (176/384 dims hold 95% of the
variance), so the decision isn't obvious from the numbers alone. Instead of
picking by rule of thumb, this cell tests it empirically: it clusters the data
**with** and **without** PCA across a range of K values and compares the
resulting cluster quality.

**What the cell does:**

- **Deduplicates first** (`np.unique`) — copy-forward repeats collapse onto the
  same point and would distort both the elbow and the PCA variance. The choice is
  made on the structure, not on the redundancy.
- **Builds two representations** — the full 384-dim embeddings, and a PCA version
  keeping 95% of the variance (~176 dims), re-normalized to restore cosine geometry.
- **Runs KMeans for each K (5 → 60)** on both representations and records the
  **silhouette score**. Silhouette is used (not inertia) because it's bounded in
  [-1, 1] and therefore comparable across different dimensionalities, whereas
  inertia is not — fewer dimensions mean smaller distances and a mechanically
  lower inertia.
- **Plots both silhouette curves on one chart** and prints each representation's
  best silhouette and the K where it occurs.

#### How to read the result

Compare the two silhouette curves:

- **Curves overlap (no clear gap)** → the representations are equivalent. *Now*
  you can invoke simplicity and skip PCA — but it's a justified decision, because
  you've shown you lose nothing by dropping it.
- **PCA clearly higher** → PCA helps; use it.
- **PCA clearly lower** → reducing dimensions threw away useful signal; skip PCA,
  also justified.

Whichever representation wins, do the final sanity check by hand: open a few
clusters and read real notes to confirm they separate in a clinically coherent
way. The metric guides the choice; the manual read confirms it.
Isto deixa a decisão documentada e sustentada por evidência — exatamente a frase que queres poder defender na tese: não "saltei o PCA por simplicidade", mas "comparei as duas representações e mostrei que eram equivalentes/piores/melhores".

In [ ]:
emb_unique = np.unique(emb, axis=0)
print(f"Unique vectors: {len(emb_unique):,} of {len(emb):,}")

reps = {
    "No PCA (384 dimensions)": emb_unique,
    "PCA 95% (~176 dimensions": normalize(
        PCA(n_components=0.95, random_state=42).fit_transform(emb_unique), norm="l2"
    ).astype(np.float32),
}
rng = np.random.RandomState(42)
idx = rng.choice(len(emb_unique), size=min(10000, len(emb_unique)), replace=False)
K_range = range(5, 61, 5)

fig, ax = plt.subplots(figsize=(7, 4))
for name, X in reps.items():
    sils = []
    for k in K_range:
        labels = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=10,
                                 batch_size=4096).fit_predict(X)
        sils.append(silhouette_score(X[idx], labels[idx]))
    ax.plot(list(K_range), sils, "o-", label=name)
    print(f"{name}: best silhouette {max(sils):.4f} at K={list(K_range)[np.argmax(sils)]}")
ax.set_xlabel("K (number of clusters)"); ax.set_ylabel("silhouette"); ax.set_title("PCA: No vs Yes"); ax.legend()
plt.tight_layout(); plt.show()